## GRIDGENE: Guided Region Identifcation based on Density of GENEs – a transcript density-based approach to characterize tissues by spatial transcriptomics ##

GRIDGENE (Guided Region Identification based on Density of GENEs) is a Python package developed for the identification and analysis of regions of interest in spatial omics data. By leveraging transcript density, GRIDGENE enables rapid and customizable exploration of tissue architecture. The tool allows users to generate tissue masks guided by prior biological knowledge, offering both flexibility and precision for spatial data analysis.

The package is freely available on GitHub: https://github.com/deMirandaLab/GRIDGENE, and a detailed description of the method can be found in the accompanying preprint on bioRxiv: https://www.biorxiv.org/content/10.1101/2025.08.14.670318v2

We demonstrate the utility of GRIDGENE by applying it to spatial transcriptomics data from the Xenium platform of colorectal cancer (CRC) samples 

Released under the GNU Public License (version 3.0).

In [ ]:
## Loading in necessary packes ## 

# -----------------------------
# IPython Extensions
# -----------------------------
%load_ext autoreload

# -----------------------------
# Standard Library
# -----------------------------
import os
import json
import sys
from glob import glob
import time
import logging
import re
from copy import deepcopy  

# -----------------------------
# Community packages
# -----------------------------
from tqdm import tqdm
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import colorcet as cc
from natsort import natsorted
from PIL import Image
import cv2
import tifffile as tiff
import anndata as ad
import scanpy as sc
import seaborn as sns
import rasterio
from rasterio.features import shapes
from shapely.geometry import Polygon, mapping, shape
from shapely.ops import unary_union
from shapely import is_valid

# -----------------------------
# Custom Modules (GRDIGENE)
# -----------------------------

from gridgene import get_arrays as ga
from gridgene import contours
from gridgene import get_masks
from gridgene.mask_properties import MaskAnalysisPipeline, MaskDefinition
from gridgene.overlay import Overlay

# -----------------------------
# Defining logger 
# -----------------------------'

logger = logging.getLogger('contour_logger')
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.setLevel(logging.INFO)

## Spatial transcriptomics data background: Xenium data from 2 colorectal cancer (CRC) samples ## 

In cancer, the main biological compartments within a tumor tissue are typically divided into epithelial regions, which are primarily composed of malignant epithelial (cancer) cells, and stromal regions, which consist of fibroblasts, endothelial cells, and diverse immune cell populations. Pathologists routinely identify these compartments based on tissue morphology using histological staining.

In spatial transcriptomics experiments, similar compartmentalization can be achieved computationally by identifying areas with high expression of gene sets characteristic of each compartment, thereby bypassing the need for explicit cell segmentation. For example, epithelial regions can be detected based on high expression of epithelial markers such as EPCAM, KRT8, and KRT18, while stromal regions are enriched for genes like COL1A1, VIM, and ACTA2.

Deriving masks for major biological compartments—such as epithelial and stromal regions—enables downstream analyses that target specific tissue domains or the interfaces between them. These interfaces often represent zones of intense biological interaction, such as immune cell infiltration or tumor invasion fronts, making them particularly relevant for understanding tumor biology and therapeutic response.

As an illustrative example, in this practical session you will apply GRIDGENE to spatial transcriptomics data generated using the Xenium platform from two distinct colorectal cancer samples. Following the workflow discussed in the lecture, you will use GRIDGENE to identify and explore biologically meaningful compartments within the tumor tissue.

Specifically, you will follow these key steps:
1. Contour identification       – detect tissue contours based on transcript density of pre-defined marker genes 
2. Mask generation              - create compartment-specific masks (e.g., epithelial and stromal masks) from the contours.
3. Mask information retrieval   - extract and summarize relevant information from each region of interest.
4. Cell segmentation overlay    - visualize and integrate cell segmentation data with the generated masks for spatial interpretation.

## == 1. Contour Identifcation ==  ##

GRIDGENE’s primary objective is to identify and outline areas of interest within spatial transcriptomics data — a process referred to as contouring. This is achieved by detecting regions with a high density of user-defined transcripts of genes of interest and generating contours around them. As discussed in the lecture, GRIDGENE currently implements three approaches for contour identification:

1. Convolution sum: A sliding window is applied across the spatial tissue coordinates, summing the expression counts of user-defined transcripts of genes of interest within each window. This produces a local density map, and regions exceeding a defined threshold are contoured to highlight transcript-dense areas
2. Neighbor counting: For each transcript, neighboring points within a user-defined radius are counted using a KD-tree algorithm. This results in per-transcript neighbor counts, enabling contouring of regions where local transcript density surpasses a specified threshold
3. SOM approach: Transcriptomic data are binned and normalized, then classified using an unsupervised Self-Organizing Map (SOM) algorithm. The classified bins are mapped back onto the tissue to identify spatial domains, without requiring predefined gene sets or threshold values.

GRIDGENE primarily relies on the convolutional sum method to identify regions of interest. In this approach, a sliding window moves across the tissue, and at each position, the number of transcripts within the window is summed. This sum is assigned to the central point of the window, generating a map of local transcript density across the tissue. Contours are then defined based on specific criteria, for example by highlighting regions where the transcript count of genes of interest exceed or fall below a user-defined threshold.

For this practical session, we will use the convolutional sum method to contour regions of interest (e.g. epithelial and stroma) and explore spatial transcript density within the colorectal cancer samples.

GRIDGENE parameter setting

GRIDGENE offers several adjustable parameters that control how contours are detected. Key parameters include kernel size, which determines the spatial scale over which transcript counts are summed, and minimum transcript counts, which sets the threshold for defining a region as dense. The tool also supports contour filtering, allowing the removal of regions based on criteria such as size or insufficient gene expression. These parameters provide control over the granularity and sensitivity of contour detection, enabling exploration of tissue regions at different levels of detail. Parameters like kernel size and density thresholds can be adapted for each spatial transcriptomics platform to reflect its unique characteristics. Additionally, these settings can be tuned to define regions at varying levels of granularity—from highly detailed, cell-level distinctions to broader, smoother masks that capture overall tissue architecture.

In this practical session, try using different values for the parameters, such as kernel size and transcript thresholds, and observe what happens to the contours. This will help you see how parameter choices affect the size, shape, and number of detected regions.

As explained above, you will now use GRIDGENE to identify the two major biological compartmentes of a tumor.
1. The epithelial compartment: primarily composed of malignant epithelial (cancer) cells.
2. The stroma compartment    : consist of fibroblasts, endothelial cells, and diverse immune cell populations

In [ ]:
## Setting GRIDGENE parameters ##

#Parameters epithelial compartment
target_genes_epithelial = ['EPCAM', 'SMIM22','CLDN3', 'KRT18','LGALS4', 'KRT8', 'ELF3','TSPAN8', 'STMN1', 'CD47', 'MYC', 'LGALS3'] 
density_th_epithelial   = 20   # 50
min_area_epithelial     = 700
kernel_size_epithelial  = 10 

#Parameters empty regions
density_th_empty = 30
min_area_th_empty = 400 
kernel_size_empty = 10

#See explanation below

Explanation GRIDGENE parameter settings

target_genes_epithelial is a list of genes whose expression is characteristic of epithelial cells in the tissue. GRIDGENE uses these genes to identify regions likely to be composed of epithelial (cancer) cells. By focusing on the density of transcripts from these specific genes, the algorithm can generate contours that define the epithelial compartment without requiring cell segmentation. Selecting an appropriate set of target genes is crucial because it directly influences the accuracy of the identified epithelial regions.

The other parameters — density_th_epithelial, min_area_epithelial, and kernel_size_epithelial — control the threshold for transcript density, the minimum size of a contour to be considered valid, and the spatial scale of the sliding window, respectively.

Similarly, parameters for empty regions define areas with low overall transcript density, helping to exclude non-tissue or sparsely populated regions from the analysis.

Now that we have set the parameters, we can start generating contours for the epithelial compartment. Begin by using the default GRIDGENE parameters to create the first set of contours. Once you have done this, experiment with changing the parameters—for example, adjusting kernel size, density threshold, or minimum area—and observe the effects.

Ask yourself: How do these changes influence the size, shape, and number of detected contours?

In [ ]:
## Loading in transcript data from the 2 colorectal cancer cores ## 
sample_files = {
    "sample_1": "data/sample_1_transcripts.csv",
    "sample_2": "data/sample_2_transcripts.csv",
}

#Load each file into a pandas dataframe and store in a dictionary
samples_dict = {name: pd.read_csv(path) for name, path in sample_files.items()}
samples_dict.items()

In [ ]:
## 1. Generating contours ##
all_contours_per_sample = {} #Initializing dictionary to store all contours per sample 
min_max_coords_per_sample = {} #Initializing dictionary to store all min coordinates per sample

for sample_name, transcripts_dataframe in samples_dict.items():
    print(f"Currently getting contours for CRC {sample_name}")
    
    df_total = transcripts_dataframe[['x_location', 'y_location', 'feature_name']]
    df_total = df_total.rename(columns={'feature_name': 'target'})
    df_total = df_total[~df_total['target'].str.contains('System|egative')]
    df_total['X'] = df_total['x_location'] - min(df_total['x_location'])
    df_total['Y'] = df_total['y_location'] - min(df_total['y_location'])

    n_genes = len(df_total['target'].unique())
    height = int(max(df_total['X'])) + 1
    width = int(max(df_total['Y'])) + 1

    min_max_coords_per_sample[sample_name] = {
        'min_x':min(df_total['x_location']),
        'max_x':max(df_total['x_location']),
        'min_y':min(df_total['y_location']),
        'max_y':max(df_total['y_location'])
    }

    print(f'Number of genes detected: {n_genes}')
    print(f'Shape: {height}, {width}')
    print(f'Numer of individual transcripts detected {len(df_total)}')

    #This makes the sparse df to an array with the spatial information 
    target_dict_total = {target: index for index, target in enumerate(df_total['target'].unique())}
    array_total = ga.transform_df_to_array(df = df_total, target_dict=target_dict_total, array_shape = (height, width,len(target_dict_total))).astype(np.int8)

    #Creating subsets
    df_subset_epi, array_subset_epi, target_indices_subset_epi = ga.get_subset_arrays(df_total, array_total,target_dict_total, target_list=target_genes_epithelial, target_col = 'target')

    #1. Epithelial compartment contours
    conts_interest_epi = contours.ConvolutionContours(array_subset_epi, contour_name = "Epithelial compartment")
    conts_interest_epi.get_conv_sum(kernel_size=kernel_size_epithelial, kernel_shape="square")
    conts_interest_epi.contours_from_sum(density_threshold = density_th_epithelial, min_area_threshold = min_area_epithelial , directionality = "higher")

    #2. Empty contours
    CEmpty = contours.ConvolutionContours(array_total, contour_name="Empty")
    CEmpty.get_conv_sum(kernel_size=kernel_size_empty, kernel_shape="square")
    CEmpty.contours_from_sum(density_threshold = density_th_empty, min_area_threshold = min_area_th_empty, directionality = "lower")

    #Store in dictionary
    all_contours_per_sample[sample_name] = {
        "Epithelial": conts_interest_epi,
        "Empty": CEmpty
    }

    ## Plotting convulution sum points and contours ##
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))

    conts_interest_epi.plot_conv_sum(cmap='plasma', c_countour='white', ax=axs[0,0])
    axs[0,0].set_title('Epithelial points and epithelial contours')
    axs[0,0].axis('off')

    CEmpty.plot_conv_sum(cmap='plasma', c_countour='white', ax=axs[0,1])
    axs[0,1].set_title('Total points and contours for empty areas')
    axs[0,1].axis('off')

    conts_interest_epi.plot_contours_scatter(path=None, show=False, s=0.05, alpha=0.5, linewidth=1, c_points= 'blue',c_contours= 'red', ax=axs[1,0])
    axs[1,0].set_title('Epithelial points and epithelial contours')
    axs[1,0].axis('off')

    CEmpty.plot_contours_scatter(path=None, show=False, s=0.05, alpha=0.5, linewidth=1, c_points= 'blue',c_contours= 'red', ax=axs[1,1])
    axs[1,1].set_title('Total points and contours for empty areas')
    axs[1,1].axis('off')

    plt.show()
    plt.close()

## == 2. Mask Generation ==  ##

The contours you have generated can now be converted into masks by GRIDGENE. In this process, contours are transformed into binary masks, which can be further refined using morphological operations to improve quality and consistency.

In this practical, you have generated contours for both the epithelial compartment and empty regions of the tissue. Epithelial contours were derived from epithelial-specific genes, while empty regions were identified using convolutional kernels to detect areas with low overall transcript density.

Using these contours, you can now generate masks for the epithelial (cancer) compartment and the stromal compartment. The cancer mask is directly defined by the epithelial contours, whereas the stroma mask is defined by negative selection, encompassing all regions that are neither cancer nor empty. Negative selection is used for the stroma because its heterogeneous composition makes it difficult to define using a universal set of marker genes.

In [ ]:
## 2. Mask generation ##
all_masks_per_sample = {} #Initializing dictionary to store all masks per sample 

for sample_name, contours_dict in all_contours_per_sample.items():
    print(f"Generating masks based on contours for {sample_name}")
    
    # Access epithelial and empty contours
    epi_contours = contours_dict["Epithelial"]
    empty_contours = contours_dict["Empty"]
    
    GM = get_masks.GetMasks(image_shape = (height, width))

    mask_empty = GM.create_mask(empty_contours.contours)
    mask_epi = GM.create_mask(epi_contours.contours)
    mask_stroma = GM.subtract_masks(np.ones((height, width), dtype=np.uint8), mask_epi, mask_empty)          
    mask_stroma = GM.filter_binary_mask_by_area(mask_stroma, min_area=700)

    #Store in dictionary
    all_masks_per_sample[sample_name] = {
        "Epithelial_mask": mask_epi,
        "Stroma_mask": mask_stroma
    }

    ##Plotting contours and masks side by side 
    fig, axs = plt.subplots(1, 3, figsize=(20, 10))
    
    epi_contours.plot_conv_sum(cmap='plasma', c_countour='white', ax=axs[0])
    axs[0].set_title('Epithelial points and epithelial contours')
    axs[0].axis('off')

    empty_contours.plot_conv_sum(cmap='plasma', c_countour='white', ax=axs[1])
    axs[1].set_title('Total points and contours for empty areas')
    axs[1].axis('off')

    GM.plot_masks(masks=[mask_epi, mask_stroma], mask_names=['Epithelium', 'Stroma'], background_color=(1, 1, 1), mask_colors={'Epithelium': (255, 165, 0), 'Stroma': (65, 105, 225)}, show=True, ax= axs[2])
    axs[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    plt.close()

In addition to delineating epithelial and stromal regions, it is often biologically informative to characterize the interface between these compartments. This interface represents the zone where cancer cells interact with stromal and immune cells, which can be critical for processes such as tumor invasion, immune infiltration, and cancer cell killing by immune cells. 

To analyze this interface, the cancer mask can be expanded into the surrounding stromal areas, creating a boundary region that captures cells in close proximity to the tumor. This expanded region can then be used for downstream analyses, such as assessing cell composition, gene expression patterns, or spatial interactions at the tumor–stroma boundary. By explicitly defining the interface, researchers can gain insights into the microenvironmental context that is often missed when analyzing cancer and stroma as separate compartments. We will expand 10 micrometers as a default, but can you try as well to use different expansion sizes and examine how this effects your analyses? 

In [ ]:
## Mask expansion ##
all_expanded_masks = {}

for sample_name, contours_dict in all_masks_per_sample.items():
    epi_mask = contours_dict["Epithelial_mask"]
    stroma_mask = contours_dict["Stroma_mask"]
    
    TA = get_masks.ConstrainedMaskExpansion(epi_mask, stroma_mask)
    TA.expand_mask(expansion_pixels=[10], min_area=1000) 

    all_expanded_masks[sample_name] = TA.binary_expansions

    mask_colors = {'expansion_10':(0, 165, 0), 
               'seed_mask':(255, 165, 0), #Epithelial compartment
               'constraint_remaining':(65, 105, 225) } #Stroma

    GM.plot_masks(masks=TA.binary_expansions.values(),
              mask_names=TA.binary_expansions.keys(),
              background_color=(1, 1, 1), mask_colors=mask_colors, path=None, show=True, ax=None,
              figsize=(8, 6))

## == 3. Mask Information Retrieval ==  ##

GRIDGENE can extract a wide range of information from masks, including transcript counts per gene and morphological features, while preserving the topological and hierarchical relationships between masks. This makes it a comprehensive tool for quantitative analysis of spatial transcriptomics data. Key types of information that can be retrieved include:
1. Transcript analyses: Extracts the number of transcripts per gene within each mask.
2. Morphological features: Measures attributes such as area, perimeter, and shape descriptors.
3. Captures hierarchical and topological relationships between original objects and their expansions, linking expanded regions back to the parent object. (Note: this feature is beyond the scope of the current practical session. More information can be found in the preprint on bioRxiv)

Let’s now proceed to extract these types of information in the code below.

In [ ]:
## Mask information retrieval ## 

mask_information_per_sample = []

for sample_name, masks_dict in all_expanded_masks.items():
    # 1. Define your masks
    mask_definitions = [
        MaskDefinition(mask=masks_dict['constraint_remaining'], mask_name='Stroma_remaining', analysis_type='bulk'),
        MaskDefinition(mask=masks_dict['seed_mask'], mask_name='Epithelium', analysis_type='bulk'),
        MaskDefinition(mask=masks_dict['expansion_10'], mask_name='Expansion_10um', analysis_type='bulk'),
    ]

    # 2. Initialize and run the analysis pipeline
    pipeline = MaskAnalysisPipeline(mask_definitions=mask_definitions, 
                                    array_counts=array_total,
                                    target_dict=target_dict_total)
    
    results = pipeline.run()
    df = pipeline.get_results_df()
    
    #Add sample information
    df['sample_name'] = sample_name
    
    #Store in the list
    mask_information_per_sample.append(df)

#Concatenate all samples into a single dataframe
mask_information_df = pd.concat(mask_information_per_sample, ignore_index=True)
mask_information_df

Now that we have extracted transcript information from the masks, we can proceed with downstream analyses to gain biological insights. One common approach is to perform differential gene expression (DGE) analysis between the different tissue compartments — namely, the epithelial, interface, and stromal regions. This allows us to identify genes that are specifically enriched in each compartment, reflecting their distinct biological functions and cellular compositions.

For instance, genes upregulated in the epithelial compartment may include those related to cell proliferation, epithelial differentiation, or oncogenic signaling, while the stromal regions may show enrichment for extracellular matrix (ECM) remodeling, fibroblast activation, or immune cell recruitment. The interface region, where tumor and stromal cells interact, is often characterized by genes associated with immune activation, cell–cell communication, and tumor killing processes.

By comparing gene expression across these compartments, researchers can uncover spatially organized transcriptional programs that underlie tumor biology. This type of analysis not only highlights the molecular distinctions between regions but can also reveal gradients or transitions in gene expression, providing insights into the tumor microenvironment.

In [ ]:
# Generating adata transcript dataframe for differential gene expression analysis#
obs_columns = ['area', 'object_id', 'mask_name','analysis_type','sample_name']
obs = mask_information_df[obs_columns]
transcript_counts_columns = mask_information_df.columns.difference(obs_columns)
transcript_counts = mask_information_df[transcript_counts_columns]

adata_transcript_per_mask = ad.AnnData(transcript_counts)
adata_transcript_per_mask.obs = obs
adata_transcript_per_mask.layers["counts"] = adata_transcript_per_mask.X.copy()

adata_transcript_per_mask.obs

In [ ]:
## Differential gene expression analysis between the epithelial compartment, stromal compartment, and the interface

#Normalizing transcript counts by the total amount of transcripts found in a mask and log transforming
sc.pp.normalize_total(adata_transcript_per_mask)
sc.pp.log1p(adata_transcript_per_mask)
adata_transcript_per_mask.raw = adata_transcript_per_mask

#Running differential gene expression analysis#
sc.tl.rank_genes_groups(adata_transcript_per_mask, "mask_name", method='wilcoxon', pts=False, use_raw=True)

#Extracting DEG results
result = adata_transcript_per_mask.uns['rank_genes_groups']
groups = result['names'].dtype.names

#Creating data frame to store DEG results
DEG_df = pd.DataFrame()
for group in groups:
        group_df = pd.DataFrame({
            'gene': result['names'][group],
            'logfoldchanges': result['logfoldchanges'][group],
            'pvals': result['pvals'][group],
            'pvals_adj': result['pvals_adj'][group],
            'scores': result['scores'][group]
        })
        group_df['mask_name'] = group
        group_df = group_df[group_df['logfoldchanges'].abs() >= 0.5]
        DEG_df = pd.concat([DEG_df, group_df], ignore_index=True)

DEG_df

Normally, to prioritize biologically relevant genes, one would rank genes by multiplying the log-fold change of a gene by the fraction of masks of a specific mask type expressing that gene across samples. This approach highlights genes that are both strongly differentially expressed and prevalent across masks.

However, due to the limited number of observations in our dataset, the p-values from the differential gene expression analysis should be interpreted with caution. Each mask type is represented by only one observation per sample, effectively providing only two replicates per group. Statistical tests such as the Wilcoxon rank-sum rely on multiple observations to estimate within-group variance and calculate the probability that observed differences occurred by chance. In typical studies, many more samples and multiple masks per group would allow reliable estimation of variability and meaningful p-values. With only one or two observations per group in our dataset, the variance cannot be properly estimated, rendering the p-values unreliable. Consequently, we focus on reporting fold-change values, which remain descriptive of relative differences in gene expression between mask types, while refraining from inferring statistical significance.

To visualize these differences, we will select the top 10 differentially expressed genes per mask based on the highest logfold changes and plot their expression patterns using a matrix plot.

In [ ]:
# Visualizaiton differential gene expression in matrixplot and boxplots # 
genes_per_mask = (DEG_df.sort_values(["mask_name", "logfoldchanges"], ascending=[True, False]).groupby("mask_name").head(10)) #Top 10 genes per cluster
top_genes_dict = genes_per_mask.groupby("mask_name")["gene"].apply(list).to_dict()
sc.pl.matrixplot(adata_transcript_per_mask, var_names=top_genes_dict, groupby='mask_name', cmap = "coolwarm", use_raw=True, standard_scale='var')

#One could of course also visualize these results with other visualizations, such as boxplots. Beyond the scope of this practical session. 

Because we have spatial information for each transcript, specifically the x and y coordinates within each core, we can map the distribution of the top differentially expressed genes to their spatial locations. This allows us to directly visualize whether these genes are indeed enriched within the specific mask regions identified in our analysis. By plotting the spatial positions of transcripts for these top genes, we can validate that the expression patterns observed in the differential gene expression analysis above were correct. 

In [ ]:
## Plotting top differentially expressed genes per mask ##
epithelium_genes = top_genes_dict["Epithelium"]
interface_genes  = top_genes_dict["Expansion_10um"]
stroma_genes     = top_genes_dict["Stroma_remaining"]

#Looping through samples
for sample_name, transcripts_dataframe in samples_dict.items():
    print(f"Currently generating spatial transcript location map for top differentially expressed genes for CRC {sample_name} per mask")
    
    df_total = transcripts_dataframe[['x_location', 'y_location', 'feature_name']]
    df_total = df_total.rename(columns={'feature_name': 'target'})
    df_total = df_total[~df_total['target'].str.contains('System|egative')]
    df_total['X'] = df_total['x_location'] - min(df_total['x_location'])
    df_total['Y'] = df_total['y_location'] - min(df_total['y_location'])

    #Getting transcripts of top genes per mask
    epithelium_top_genes_df = df_total[df_total['target'].isin(epithelium_genes)]
    interface_top_genes_df  = df_total[df_total['target'].isin(interface_genes)]
    stroma_top_genes_df     = df_total[df_total['target'].isin(stroma_genes)]

    fig, axs = plt.subplots(2, 2, figsize=(15, 10))

    #1. Combined visualization
    sns.scatterplot(data=epithelium_top_genes_df, x="X", y="Y", color= (255/255, 165/255, 0/255), s=3, legend=False, ax=axs[0,0])
    sns.scatterplot(data=interface_top_genes_df, x="X", y="Y", color= (100/255, 200/255, 0/255), s=3, legend=False, ax=axs[0,0])
    sns.scatterplot(data=stroma_top_genes_df, x="X", y="Y", color= (65/255, 105/255, 225/255), s=3, legend=False, ax=axs[0,0])
    axs[0,0].set_title('Top genes epithelium, expansion, and stroma combined')

    #2. Epithelium only
    sns.scatterplot(data=epithelium_top_genes_df, x="X", y="Y", color= (255/255, 165/255, 0/255), s=3, legend=False, ax=axs[0,1])
    axs[0,1].set_title('Top genes epithelium')

    #3. Expansion only
    sns.scatterplot(data=interface_top_genes_df, x="X", y="Y", color= (100/255, 200/255, 0/255), s=3, legend=False, ax=axs[1,0])
    axs[1,0].set_title('Top genes expansion interface')

    #4. Stroma only
    sns.scatterplot(data=stroma_top_genes_df, x="X", y="Y", color= (65/255, 105/255, 225/255), s=3, legend=False, ax=axs[1,1])
    axs[1,1].set_title('Top genes remaining stroma')
    plt.show()
    plt.close()


We have now visualized the spatial distribution of the top differentially expressed genes across the two samples. Even with just these two samples, we can already observe clear enrichment of each top gene within its corresponding mask region. However, the limited number of samples included in this practical represents a constraint on the amount of data we could analyze. With additional cores, the spatial patterns and distinctions between masks would likely become even more pronounced, providing even stronger validation of the region-specific expression of these genes.

## == 4. Cell segmentation overlay ==  ##

Defining regions of interest using GRIDGENE-generated masks does not preclude the use of cell segmentation; in fact, the two approaches can be seamlessly integrated to enhance downstream analyses. By overlaying masks onto segmented cell data, individual cells can be assigned spatial labels corresponding to specific regions. For example, identifying cells as intraepithelial (within epithelial compartments), stromal, or located at the interface between compartments.

While traditional cell segmentation-based approaches can achieve similar labeling, they often rely on defining continuous neighborhoods or adjacency rules, which can be computationally complex and less intuitive, especially in heterogeneous tissues. In contrast, the use of predefined masks provides a clear and straightforward framework for assigning spatial region location for cells, reducing computational overhead and manual effort. This integration allows researchers to combine the high-resolution insights of cell segmentation with the contextual information provided by region-specific masks. 

In [ ]:
## Loading in adata data file containing cell type annotation
sample_files_cell_type_annotation = {
    "sample_1": "data/sample_1_adata_cells.h5ad",
    "sample_2": "data/sample_2_adata_cells.h5ad",
}

#Load each file and add to dictionary
samples_cell_type_annotation_dict = {name: sc.read_h5ad(path) for name, path in sample_files_cell_type_annotation.items()}

Now that we have generated masks for the epithelial compartment, the expansion interface, and the remaining stromal regions, we can integrate these masks with our cell segmentation data and corresponding cell type annotation. Overlaying the masks onto the segmented cells allows us to assign spatial labels to each cell based on the compartment it resides in — for example, identifying cells as intraepithelial, stromal, or located at the epithelial–stromal interface. This approach enables a precise mapping of cell types to their spatial context, facilitating downstream analyses such as differential expression per compartment or spatial neighborhood characterization. We will first visualize the overlay between the cell segmentation and out GRIDGENE masks below. 

In [ ]:
## Generating cell segmentation and GRIDGENE mask overlay visualization ## 
sample_files_cell_segmentation_polygons = {
    "sample_1": "data/sample_1_cell_polygons.json",
    "sample_2": "data/sample_2_cell_polygons.json",
}

#Loading in cell segmentation polygons 
samples_cell_segmentation_polygons_dict = {}
for name, path in sample_files_cell_segmentation_polygons.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        samples_cell_segmentation_polygons_dict[name] = data

for sample_name, cell_polygons in samples_cell_segmentation_polygons_dict.items():
    
    print(f"Currently generating cell segmentation and GRIDGENE mask overlay for CRC {sample_name}")

    #Getting all masks in a dictionary
    mask_dict = all_expanded_masks[sample_name]

    #Getting cell polygons in correct format
    cell_polygons_input = {"geometries": cell_polygons}

    #Getting min x and y coordinates
    min_x = min_max_coords_per_sample[sample_name]['min_x']
    min_y = min_max_coords_per_sample[sample_name]['min_y']

    #Generating the overlay
    ov = Overlay(mask_dict = mask_dict, segmentation = cell_polygons_input , segmentation_type="polygons", save_path=None, 
            min_x=min_x, min_y=min_y, flip_masks=True)

    ov.plot_masks_overlay_segmentation(colors=[(100/255, 200/255, 0/255), (255/255, 165/255, 0/255), (65/255, 105/255, 225/255)], titles=['Expansion_10um', 'Epithelium', 'Stroma_remaining'])

In the visualization, the overlay between the cell segmentation and the GRIDGENE-generated masks is now apparent. For example, the “expansion interface” region corresponds roughly to a single layer of cells surrounding the epithelial compartment. If you want to include a broader region of cells in the expansion interface, you can increase the expansion distance by rerunning the previous code with a larger value, such as 20 or 30 pixels. Doing so will expand the interface outward, encompassing more neighboring cells and providing a thicker layer around the epithelial compartment. Observe how the additional cells are incorporated and how the interface region changes in size and shape—this can help you tune the mask expansion to match the spatial resolution or biological scale you are interested in.

Now, we would like to assign cells according to the GRIDGENE mask they fall into. By combining this information with the cell type annotations stored in adata_cells, we can examine the composition of different cell types within each mask region. Moreover, this setup opens the door for more detailed analyses: for instance, one could ask how a T cell located in the expansion interface differs in gene expression from a T cell in the remaining stroma. While performing differential gene expression analyses of specific cell types between masks is beyond the scope of this practical, we will quickly explore the cell type composition per GRIDGENE mask for each sample.

In [ ]:
## Annotating cells in adata_cells by the GRIDGENE mask region they fall into ## 
annotated_adatas = []
samples_cell_segmentation_polygons_dict = {}

#Loading in cell segmentation polygons 
for name, path in sample_files_cell_segmentation_polygons.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        samples_cell_segmentation_polygons_dict[name] = data

for sample_name, cell_polygons in samples_cell_segmentation_polygons_dict.items():
    print(f"Currently annotating cells according to their GRIDGENE region mask for CRC {sample_name}")

    #Getting all masks in a dictionary
    mask_dict = all_expanded_masks[sample_name]
    
    #Getting min x and y coordinates4
    min_x = min_max_coords_per_sample[sample_name]['min_x']
    min_y = min_max_coords_per_sample[sample_name]['min_y']

    #1. Epithelium multipolygon
    polygons_epithelium = shapes(mask_dict["seed_mask"].astype(np.uint8), transform=rasterio.transform.Affine.translation(min_x, min_y))
    polygons_epithelium_final = [shape(pol_shape[0]) for pol_shape in polygons_epithelium if pol_shape[1] == 1] 
    multi_polygons_epithelium =  unary_union(polygons_epithelium_final)

    #2. Expansion multipolygon
    polygons_expansion = shapes(mask_dict["expansion_10"].astype(np.uint8), transform=rasterio.transform.Affine.translation(min_x, min_y))
    polygons_expansion_final = [shape(pol_shape[0]) for pol_shape in polygons_expansion if pol_shape[1] == 1] 
    multi_polygons_expansion =  unary_union(polygons_expansion_final)

    #3. Stroma multipolygon
    polygons_stroma = shapes(mask_dict["constraint_remaining"].astype(np.uint8), transform=rasterio.transform.Affine.translation(min_x, min_y))
    polygons_stroma_final = [shape(pol_shape[0]) for pol_shape in polygons_stroma if pol_shape[1] == 1] 
    multi_polygons_stroma =  unary_union(polygons_stroma_final)

    #Storing multipolygon of each GRIDGENE mask into a dictionary so we can loop through it 
    regions_multipolygons = {'Epithelium': multi_polygons_epithelium, 
        'Expansion_10um': multi_polygons_expansion, 
        'Stroma_remaining': multi_polygons_stroma, 
    }

    #Retrieving adata frame that contains the cell type annotation of each cell
    adata_sample = samples_cell_type_annotation_dict[sample_name]
    cells_regions = pd.DataFrame(columns=["Regions"], index=adata_sample.obs_names)

    #Checking for each cell in which GRIDGENE mask it falls. Each cell gets assigned to the region with which it intersects >= 50%
    polygon_counter = 1
    for polygon in cell_polygons:
        print(f"Processing polygon {polygon_counter} of {len(cell_polygons)}", end='\r')  # progress tracking
        cell_id = polygon["cell"]
        poly_c = Polygon(polygon["coordinates"][0])

        if is_valid(poly_c):
            for regionName, regionPolygon in regions_multipolygons.items():
                if regionPolygon.intersection(poly_c).area / poly_c.area >=0.5:
                    cells_regions.at[cell_id, "Regions"] = regionName
                    break

        else:
            if poly_c.area > 0:
                for regionName, regionPolygon in regions_multipolygons.items():
                    if regionPolygon.intersection(poly_c.buffer(0)).area / poly_c.area >= 0.5:
                        cells_regions.at[cell_id, "Regions"] = regionName
                        break
        polygon_counter += 1

    adata_sample.obs["GRIDGENE_mask"] = cells_regions["Regions"]
    annotated_adatas.append(adata_sample)

merged_adata = ad.concat(annotated_adatas)
merged_adata.obs

If we now inspect merged_adata, we can see that the two CRC samples have been successfully combined into a single AnnData object. For each cell, we know its original sample, its cell type, and the GRIDGENE mask region it belongs to. As described earlier, this allows us to analyze the cell type composition within specific GRIDGENE regions, as well as investigate region-specific functional differences among cells of the same type.

In [ ]:
## Cell type composition per GRIDGENE mask ## 

#Create a nested dictionary to store results
composition_per_sample = {}

for sample in merged_adata.obs["SectionID"].unique():
    #Subset the adata to this sample
    adata_sample = merged_adata[merged_adata.obs["SectionID"] == sample]
    
    #Count cells per GRIDGENE_mask and per Cell_type
    counts = pd.crosstab(
        adata_sample.obs["GRIDGENE_mask"],
        adata_sample.obs["Cell_type"]
    )
    
    #Calculate fractions per mask
    fractions = counts.div(counts.sum(axis=1), axis=0)
    
    #Store in dictionary
    composition_per_sample[sample] = {
        "counts": counts,
        "fractions": fractions
    }

composition_per_sample

Now, lets visualize the cell type composition in stacked bargraphs. 

In [ ]:
## Visualizing cell type composition per GRIDGENE mask ## 

# Get all unique cell types
all_cell_types = sorted(merged_adata.obs['Cell_type'].unique())

# Create a color palette (Set3 is softer, visually nicer)
cmap = plt.get_cmap('tab20')
colors = {cell_type: cmap(i % cmap.N) for i, cell_type in enumerate(all_cell_types)}

#Precompute color list to use in plots
color_list = [colors[cell] for cell in all_cell_types]

#Loop through samples
for sample_id in merged_adata.obs["SectionID"].unique():
    fractions = composition_per_sample[sample_id]["fractions"]

    #Ensure all cell types are present
    fractions = fractions.reindex(columns=all_cell_types, fill_value=0)
    
    ax = fractions.plot(
        kind='bar',
        stacked=True,
        figsize=(8, 5),
        color=color_list  #consistent pretty colors
    )
    
    plt.ylabel("Fraction of cells")
    plt.xlabel("GRIDGENE mask")
    plt.title(f"Cell type composition per GRIDGENE mask ({sample_id})")
    plt.legend(title="Cell type", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

When examining the cell type composition plot of Sample 1, distinct spatial patterns between regions are immediately apparent. Notably, T cells and macrophages appear to be enriched at the tumor border, whereas B cells and plasma cells are more abundant in the surrounding stroma. This distribution aligns with their expected spatial biology: plasma cells are primarily responsible for antibody production in the stromal compartment, while some of the T cells (e.g. CD8+ T cells) are likely engaged in cytotoxic activity targeting tumor cells. 

Moreover, the plots reveal clear intra-tumoral heterogeneity across samples. For example, Sample 2 contains a substantially higher proportion of fibroblasts compared to Sample 1 and appears to be overall less infiltrated by immune cells. This suggests that Sample 2 may have a more fibrotic stroma with limited immune surveillance, which could have implications for tumor progression and response to immunotherapy. These observations underscore the importance of spatial context in understanding the composition and functional state of the tumor microenvironment.